# Movie Recommender — Training Notebook

Content-based recommender over **4,806 films**. This notebook documents the full pipeline:
raw TMDB-5000 data → NLP feature engineering → `tags` → `CountVectorizer` → cosine similarity.

The recommender models each movie as a bag-of-words vector built from its genres, keywords,
top-3 cast, director and overview, then ranks the most similar titles by cosine similarity.


## 1. Feature engineering (reference)

The `tags` column shipped in `movies_dict.pkl` was produced from the two TMDB-5000 CSVs
(`tmdb_5000_movies.csv`, `tmdb_5000_credits.csv`) using the steps below. Those raw CSVs are
available on Kaggle. This cell documents *how* `tags` were engineered; the notebook then
reproduces the model from the shipped `movies_dict.pkl` so it runs without the raw files.


In [ ]:
# --- Reference pipeline that produced `tags` (needs the raw TMDB-5000 CSVs) ---
# import ast
# movies = pd.read_csv('tmdb_5000_movies.csv')
# credits = pd.read_csv('tmdb_5000_credits.csv')
# df = movies.merge(credits, on='title')
# df = df[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
# df.dropna(inplace=True)
#
# def names(obj):                       # genres / keywords -> list of names
#     return [i['name'] for i in ast.literal_eval(obj)]
# def top_cast(obj):                     # first 3 billed actors
#     return [i['name'] for i in ast.literal_eval(obj)[:3]]
# def director(obj):
#     return [i['name'] for i in ast.literal_eval(obj) if i['job'] == 'Director']
#
# df['genres']   = df['genres'].apply(names)
# df['keywords'] = df['keywords'].apply(names)
# df['cast']     = df['cast'].apply(top_cast)
# df['crew']     = df['crew'].apply(director)
# df['overview'] = df['overview'].apply(lambda x: x.split())
# # collapse multi-word names so 'Sam Worthington' -> one token 'samworthington'
# for col in ['genres', 'keywords', 'cast', 'crew']:
#     df[col] = df[col].apply(lambda xs: [x.replace(' ', '') for x in xs])
# df['tags'] = df['overview'] + df['genres'] + df['keywords'] + df['cast'] + df['crew']
# df['tags'] = df['tags'].apply(lambda xs: ' '.join(xs).lower())
#
# # Porter-stem every token
# from nltk.stem.porter import PorterStemmer
# ps = PorterStemmer()
# df['tags'] = df['tags'].apply(lambda t: ' '.join(ps.stem(w) for w in t.split()))
# movies = df[['movie_id', 'title', 'tags']].rename(columns={'movie_id': 'movie_id_x'})

## 2. Load the processed dataset

`movies_dict.pkl` holds the output of the pipeline above: `movie_id_x` (TMDB id), `title`,
and the cleaned, Porter-stemmed `tags` string.


In [ ]:
import pickle
import pandas as pd

movies = pd.DataFrame(pickle.load(open('movies_dict.pkl', 'rb')))
print(movies.shape)
movies.head(3)

In [ ]:
# A single tags document = the NLP fingerprint of one movie
movies['tags'].iloc[0][:300]

## 3. Vectorize the tags

Bag-of-words over the 5,000 most frequent tokens (English stop-words removed).
Each movie becomes a 5,000-dim count vector.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(movies['tags']).toarray()
print('vectors:', vectors.shape, '| vocabulary size:', len(cv.vocabulary_))

## 4. Cosine similarity

Cosine similarity between every pair of movie vectors → a 4806×4806 matrix.
This is the model artifact the app queries at runtime (recomputed on load, not shipped as a pickle).


In [ ]:
import time
from sklearn.metrics.pairwise import cosine_similarity

start = time.perf_counter()
similarity = cosine_similarity(vectors)
print('build time: %.2fs' % (time.perf_counter() - start))
print('matrix:', similarity.shape)

## 5. Recommend

For a chosen title, sort its similarity row and take the next 5 most similar movies.


In [ ]:
def recommend(title, n=5):
    idx = movies.index[movies['title'] == title][0]
    ranked = sorted(enumerate(similarity[idx]), reverse=True, key=lambda x: x[1])[1:n + 1]
    return [movies.iloc[i].title for i, _ in ranked]

recommend('Avatar')

In [ ]:
recommend('The Dark Knight')

## 6. CountVectorizer vs TF-IDF

TF-IDF down-weights tokens that appear across many movies (e.g. common genres) and up-weights
distinctive ones. Below we compare the top-5 for the same title under both vectorizers.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf_vectors = tfidf.fit_transform(movies['tags'])
tfidf_sim = cosine_similarity(tfidf_vectors)

def recommend_tfidf(title, n=5):
    idx = movies.index[movies['title'] == title][0]
    ranked = sorted(enumerate(tfidf_sim[idx]), reverse=True, key=lambda x: x[1])[1:n + 1]
    return [movies.iloc[i].title for i, _ in ranked]

print('CountVectorizer:', recommend('Avatar'))
print('TF-IDF        :', recommend_tfidf('Avatar'))

## 7. Export the artifact

The app loads `movies_dict.pkl` and recomputes `similarity` at startup (see `app.py`),
so no large matrix pickle is committed. Re-export the dict here if the pipeline changes.


In [ ]:
# movies.to_dict() is what app.py loads via pickle
# pickle.dump(movies.to_dict(), open('movies_dict.pkl', 'wb'))
print('movies_dict.pkl already present:', movies.shape)